# Feature Engineering

---

1. Import packages
2. Load data
3. Feature engineering

---

## 1. Environment Setup & Data Loading

In [2]:
# --- STEP 1: ENVIRONMENT SETUP & DATA LOADING ---

# Import core libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Set visual styling
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)
pd.set_option('display.max_columns', None)

---
## 2. Load data

In [3]:
# Load the updated clean dataset after EDA and the raw price dataset
clean_data = pd.read_csv('clean_data_after_eda.csv')
price_data = pd.read_csv('price_data.csv')

# Verify the datasets loaded correctly
print("=== Clean Data Preview ===")
display(clean_data.head(3))
print(f"clean_data shape: {clean_data.shape}")

print("\n=== Price Data Preview ===")
display(price_data.head(3))
print(f"price_data shape: {price_data.shape}")

=== Clean Data Preview ===


,id,channel_sales,cons_12m,cons_gas_12m,cons_last_month,date_activ,date_end,date_modif_prod,date_renewal,forecast_cons_12m,forecast_cons_year,forecast_discount_energy,forecast_meter_rent_12m,forecast_price_energy_off_peak,forecast_price_energy_peak,forecast_price_pow_off_peak,has_gas,imp_cons,margin_gross_pow_ele,margin_net_pow_ele,nb_prod_act,net_margin,num_years_antig,origin_up,pow_max,var_year_price_off_peak_var,var_year_price_peak_var,var_year_price_mid_peak_var,var_year_price_off_peak_fix,var_year_price_peak_fix,var_year_price_mid_peak_fix,var_year_price_off_peak,var_year_price_peak,var_year_price_mid_peak,var_6m_price_off_peak_var,var_6m_price_peak_var,var_6m_price_mid_peak_var,var_6m_price_off_peak_fix,var_6m_price_peak_fix,var_6m_price_mid_peak_fix,var_6m_price_off_peak,var_6m_price_peak,var_6m_price_mid_peak,churn
0,24011ae4ebbe3035111d65fa7c15bc57,foosdfpfkusacimwkcsosbicdxkicaua,0,54946,0,2013-06-15,2016-06-15,2015-11-01,2015-06-23,0.00,0,0.0,1.78,0.114481,0.098142,40.606701,t,0.0,25.44,25.44,2,678.99,3,lxidpiddsbxsbosboudacockeimpuepw,43.648,0.000061,2.627605e-05,0.00044,1.102785,49.550703,22.022535,1.102846,4.955073e+01,22.022975,0.000131,4.100838e-05,0.000908,2.086294,99.530517,44.235794,2.086425,9.953056e+01,44.236702,1
1,d29c2c54acc38ff3c0614d0a653813dd,MISSING,4660,0,0,2009-08-21,2016-08-30,2009-08-21,2015-08-31,189.95,0,0.0,16.27,0.145711,0.000000,44.311378,f,0.0,16.38,16.38,1,18.89,6,kamkkxfxxuwbdslkwifmmcsiusiuosws,13.800,0.000005,6.089453e-04,0.00000,0.006465,0.000000,0.000000,0.006470,6.089453e-04,0.000000,0.000003,1.217891e-03,0.000000,0.009482,0.000000,0.000000,0.009485,1.217891e-03,0.000000,0
2,764c75f661154dac3a6c254cd082ea7d,foosdfpfkusacimwkcsosbicdxkicaua,544,0,0,2010-04-16,2016-04-16,2010-04-16,2015-04-17,47.96,0,0.0,38.72,0.165794,0.087899,44.311378,f,0.0,28.60,28.60,1,6.60,6,kamkkxfxxuwbdslkwifmmcsiusiuosws,13.856,0.000006,2.558511e-07,0.00000,0.007662,0.000000,0.000000,0.007668,2.558511e-07,0.000000,0.000004,9.450150e-08,0.000000,0.000000,0.000000,0.000000,0.000004,9.450150e-08,0.000000,0


clean_data shape: (14606, 44)

=== Price Data Preview ===


,id,price_date,price_off_peak_var,price_peak_var,price_mid_peak_var,price_off_peak_fix,price_peak_fix,price_mid_peak_fix
0,038af19179925da21a25619c5a24b745,2015-01-01,0.151367,0.0,0.0,44.266931,0.0,0.0
1,038af19179925da21a25619c5a24b745,2015-02-01,0.151367,0.0,0.0,44.266931,0.0,0.0
2,038af19179925da21a25619c5a24b745,2015-03-01,0.151367,0.0,0.0,44.266931,0.0,0.0


price_data shape: (193002, 8)


## 3. Feature Engineering

### Analyzing Price Trends Across Months (Estelle's Hypothesis)

In [4]:
# --- STEP 2: ANALYZING PRICE TRENDS ACROSS MONTHS (ESTELLE'S HYPOTHESIS) ---

# 1. Ensure price_date is in proper datetime format
price_data['price_date'] = pd.to_datetime(price_data['price_date'], errors='coerce')

# 2. Extract year and month for time-based filtering
price_data['price_year'] = price_data['price_date'].dt.year
price_data['price_month'] = price_data['price_date'].dt.month

# 3. Isolate historical pricing data to examine off-peak prices specifically (variable & fixed)
off_peak_prices = price_data[['id', 'price_date', 'price_year', 'price_month', 'price_off_peak_var', 'price_off_peak_fix']]

print("=== Off-Peak Price Subset Preview ===")
display(off_peak_prices.head(3))

# 4. Filter for December (Month 12) and January (Month 1) to compare preceding price patterns
dec_jan_prices = off_peak_prices[off_peak_prices['price_month'].isin([12, 1])]

print(f"\nFiltered records for December and January: {len(dec_jan_prices)}")
display(dec_jan_prices.head(3))

=== Off-Peak Price Subset Preview ===


,id,price_date,price_year,price_month,price_off_peak_var,price_off_peak_fix
0,038af19179925da21a25619c5a24b745,2015-01-01,2015,1,0.151367,44.266931
1,038af19179925da21a25619c5a24b745,2015-02-01,2015,2,0.151367,44.266931
2,038af19179925da21a25619c5a24b745,2015-03-01,2015,3,0.151367,44.266931



Filtered records for December and January: 32164


,id,price_date,price_year,price_month,price_off_peak_var,price_off_peak_fix
0,038af19179925da21a25619c5a24b745,2015-01-01,2015,1,0.151367,44.266931
11,038af19179925da21a25619c5a24b745,2015-12-01,2015,12,0.145859,44.444710
12,31f2ce549924679a3cbb2d128ae9ea43,2015-01-01,2015,1,0.125976,40.565969


### Engineering the Core Price Difference Feature

In [5]:
# --- STEP 3: ENGINEERING THE CORE PRICE DIFFERENCE FEATURE ---

# 1. Pivot or filter the December and January off-peak variable prices per customer
dec_prices = price_data[price_data['price_month'] == 12][['id', 'price_off_peak_var']].rename(columns={'price_off_peak_var': 'price_off_peak_dec'})
jan_prices = price_data[price_data['price_month'] == 1][['id', 'price_off_peak_var']].rename(columns={'price_off_peak_var': 'price_off_peak_jan'})

# Aggregate by customer ID in case there are multiple entries per month
dec_mean = dec_prices.groupby('id').mean().reset_index()
jan_mean = jan_prices.groupby('id').mean().reset_index()

# 2. Merge December and January prices together
price_diff_df = pd.merge(dec_mean, jan_mean, on='id', how='inner')

# 3. Calculate the differential (change) in off-peak prices between December and January
price_diff_df['off_peak_dec_jan_price_diff'] = price_diff_df['price_off_peak_dec'] - price_diff_df['price_off_peak_jan']

# Isolate the new feature column
price_diff_feature = price_diff_df[['id', 'off_peak_dec_jan_price_diff']]

print("=== Engineered Feature Preview ===")
display(price_diff_feature.head(3))

# 4. Merge this new feature into our clean dataset
clean_data = pd.merge(clean_data, price_diff_feature, on='id', how='left')

print(f"\nUpdated clean_data shape after adding feature: {clean_data.shape}")

=== Engineered Feature Preview ===


,id,off_peak_dec_jan_price_diff
0,0002203ffbb812588b632b9e628cc38d,-0.006192
1,0004351ebdd665e6ee664792efc4fd13,-0.004104
2,0010bcc39e42b3c2131ed2ce55246e3c,0.050443



Updated clean_data shape after adding feature: (14606, 45)


### Creating Additional Creative Features

In [7]:
# --- STEP 4 (CORRECTED): CREATING ADDITIONAL CREATIVE FEATURES ---

# 1. Explicitly convert date columns to datetime in clean_data
date_cols = ['date_activ', 'date_end', 'date_modif_prod', 'date_renewal']
for col in date_cols:
    if col in clean_data.columns:
        clean_data[col] = pd.to_datetime(clean_data[col], errors='coerce')

# 2. Customer Tenure (duration between activation date and contract end date in years)
if 'date_activ' in clean_data.columns and 'date_end' in clean_data.columns:
    clean_data['tenure_years'] = (clean_data['date_end'] - clean_data['date_activ']).dt.days / 365.25
    print("✓ Created feature: 'tenure_years'")

# 3. Peak vs. Off-Peak Price Ratios (using forecast prices)
if 'forecast_price_energy_peak' in clean_data.columns and 'forecast_price_energy_off_peak' in clean_data.columns:
    # Adding a small epsilon (1e-6) to prevent division by zero errors
    clean_data['forecast_peak_offpeak_ratio'] = clean_data['forecast_price_energy_peak'] / (clean_data['forecast_price_energy_off_peak'] + 1e-6)
    print("✓ Created feature: 'forecast_peak_offpeak_ratio'")

# 4. Consumption Trend Ratio (Last month consumption vs. average monthly consumption over the year)
if 'cons_last_month' in clean_data.columns and 'cons_12m' in clean_data.columns:
    avg_monthly_cons = clean_data['cons_12m'] / 12.0
    clean_data['cons_last_vs_avg_ratio'] = clean_data['cons_last_month'] / (avg_monthly_cons + 1e-6)
    print("✓ Created feature: 'cons_last_vs_avg_ratio'")

# Preview the newly created features
print("\n=== Preview of Engineered Features ===")
feature_cols_to_show = ['id', 'tenure_years', 'forecast_peak_offpeak_ratio', 'cons_last_vs_avg_ratio']
existing_preview_cols = [col for col in feature_cols_to_show if col in clean_data.columns]
display(clean_data[existing_preview_cols].head(3))

print(f"\nFinal dataset shape after all feature engineering: {clean_data.shape}")

✓ Created feature: 'tenure_years'
✓ Created feature: 'forecast_peak_offpeak_ratio'
✓ Created feature: 'cons_last_vs_avg_ratio'

=== Preview of Engineered Features ===


,id,tenure_years,forecast_peak_offpeak_ratio,cons_last_vs_avg_ratio
0,24011ae4ebbe3035111d65fa7c15bc57,3.000684,0.857270,0.0
1,d29c2c54acc38ff3c0614d0a653813dd,7.025325,0.000000,0.0
2,764c75f661154dac3a6c254cd082ea7d,6.001369,0.530167,0.0



Final dataset shape after all feature engineering: (14606, 48)


### Merging and Consolidating the Dataset

In [9]:
# --- STEP 5 (UPDATED): MERGING, CONSOLIDATING & EXPORTING THE DATASET ---

# 1. Verify all engineered features are present in clean_data
expected_features = [
    'off_peak_dec_jan_price_diff', 
    'tenure_years', 
    'forecast_peak_offpeak_ratio', 
    'cons_last_vs_avg_ratio'
]

print("=== FEATURE CONSOLIDATION CHECK ===")
for feature in expected_features:
    if feature in clean_data.columns:
        print(f"✓ Feature '{feature}' is successfully included.")
    else:
        print(f"✗ Warning: Feature '{feature}' is missing!")

# 2. Handle any potential infinite values or NaNs resulting from divisions/diffs
clean_data.replace([np.inf, -np.inf], np.nan, inplace=True)

# Fill any new NaNs with direct assignment to avoid the FutureWarning
for col in expected_features:
    if col in clean_data.columns and clean_data[col].isnull().sum() > 0:
        median_val = clean_data[col].median()
        clean_data[col] = clean_data[col].fillna(median_val)
        print(f"Imputed missing values in '{col}' with median: {median_val}")

# 3. Export the finalized feature-engineered dataset
output_filename = 'clean_data_after_feature_engineering.csv'
clean_data.to_csv(output_filename, index=False)

print(f"\nSuccessfully saved consolidated dataset to '{output_filename}'!")
print(f"Final Dataset Shape (Rows, Columns): {clean_data.shape}")

# Preview the final columns
display(clean_data[['id', 'churn'] + expected_features].head(3))

=== FEATURE CONSOLIDATION CHECK ===
✓ Feature 'off_peak_dec_jan_price_diff' is successfully included.
✓ Feature 'tenure_years' is successfully included.
✓ Feature 'forecast_peak_offpeak_ratio' is successfully included.
✓ Feature 'cons_last_vs_avg_ratio' is successfully included.

Successfully saved consolidated dataset to 'clean_data_after_feature_engineering.csv'!
Final Dataset Shape (Rows, Columns): (14606, 48)


,id,churn,off_peak_dec_jan_price_diff,tenure_years,forecast_peak_offpeak_ratio,cons_last_vs_avg_ratio
0,24011ae4ebbe3035111d65fa7c15bc57,1,0.020057,3.000684,0.857270,0.0
1,d29c2c54acc38ff3c0614d0a653813dd,0,-0.003767,7.025325,0.000000,0.0
2,764c75f661154dac3a6c254cd082ea7d,0,-0.004670,6.001369,0.530167,0.0


### Final Validation & Export

In [10]:
# --- STEP 6: FINAL VALIDATION & EXPORT ---

# 1. Check for any remaining missing values across the entire dataset
total_missing = clean_data.isnull().sum().sum()
print(f"Total missing values in dataset: {total_missing}")

# 2. Check for any remaining infinite values
total_infs = np.isinf(clean_data.select_dtypes(include=[np.number])).sum().sum()
print(f"Total infinite values in dataset: {total_infs}")

# 3. Final verification of dataset dimensions
print(f"Final Enriched Dataset Shape: {clean_data.shape[0]} rows, {clean_data.shape[1]} columns")

# 4. Save/Confirm export
output_filename = 'clean_data_after_feature_engineering.csv'
clean_data.to_csv(output_filename, index=False)
print(f"✓ Successfully verified and exported to '{output_filename}' ready for machine learning modeling!")

Total missing values in dataset: 0
Total infinite values in dataset: 0
Final Enriched Dataset Shape: 14606 rows, 48 columns
✓ Successfully verified and exported to 'clean_data_after_feature_engineering.csv' ready for machine learning modeling!
